# 01 — Exploratory Data Analysis

Chapter-3 EDA notebook for the master's thesis *«Разработка интеллектуальной системы прогнозирования времени выполнения и планирования очередей CI/CD на основе ML»*.

**Inputs:** live PostgreSQL — connects via the same DSN the api-gateway uses.
**Outputs:** PNG figures into `docs/thesis/figures/`, picked up by Chapter 3 of the manuscript.

## How to run

```bash
docker compose -f docker-compose.yml -f docker-compose.dev.yml exec ml \
  python -m jupyter nbconvert --to notebook --execute /ml/notebooks/01_eda.ipynb \
  --output 01_eda_executed.ipynb
```

Figures land in the host-mounted `./docs/thesis/figures/` so they show up in the manuscript without any copying.

## Sections

1. Dataset summary
2. Duration distribution (log-binned histogram)
3. Top job names by frequency
4. Branch breakdown
5. Hour-of-day / day-of-week patterns
6. Feature correlation matrix

In [ ]:
import os
import sys
import pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

# Reuse the same loader the training pipeline uses — guarantees we EDA
# on the exact dataframe the models train on.
sys.path.insert(0, '/app')
from app.storage.db import load_jobs_df  # noqa: E402

DSN = os.environ.get('POSTGRES_DSN', 'postgresql://cicdml:cicdml@db:5432/cicdml')
FIG_DIR = pathlib.Path('/var/lib/cicdml/thesis/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Editorial styling — match the dark dashboard palette so figures don't
# clash with screenshots in the manuscript. Light-on-white is left to
# the LaTeX side if needed.
plt.rcParams.update({
    'figure.facecolor': '#0E0F11',
    'axes.facecolor':   '#16181C',
    'axes.edgecolor':   '#2E323A',
    'axes.labelcolor':  '#ECEDEE',
    'axes.titlecolor':  '#ECEDEE',
    'xtick.color':      '#9BA1A6',
    'ytick.color':      '#9BA1A6',
    'text.color':       '#ECEDEE',
    'grid.color':       '#23262C',
    'font.family':      'monospace',
    'font.size':        10,
})
ACCENT = '#F2C94C'
INFO   = '#60A5FA'
OK     = '#4ADE80'
WARN   = '#FBBF24'
ERR    = '#F87171'

## 1. Dataset summary

In [ ]:
df = load_jobs_df(DSN)
print(f'Rows: {len(df):,}')
print(f'Repos: {df["repo_id"].nunique()}')
print(f'Date range: {df["run_created_at"].min()} → {df["run_created_at"].max()}')
df.head(3)

## 2. Duration distribution

Log-binned histogram — CI durations span 4+ orders of magnitude.
Linear axes hide all the action near the origin under a single dot.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = df['duration_sec'].clip(lower=1)
bins = np.logspace(0, np.log10(x.max() + 1), 50)
ax.hist(x, bins=bins, color=ACCENT, edgecolor='#16181C')
ax.set_xscale('log')
ax.set_xlabel('duration (sec, log scale)')
ax.set_ylabel('count')
ax.set_title(f'CI job duration distribution (n = {len(df):,})')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_3_1_duration_distribution.png', dpi=144)
plt.close(fig)

## 3. Top-K job names by frequency

In [ ]:
top = df['job_name'].value_counts().head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(len(top)), top.values, color=INFO)
ax.set_yticks(range(len(top)))
ax.set_yticklabels([t[:40] for t in top.index])
ax.set_xlabel('runs')
ax.set_title('Top-15 job_name by frequency')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_3_2_top_jobs.png', dpi=144)
plt.close(fig)

## 4. Branch breakdown — mean duration by branch class

In [ ]:
branch_map = df['head_branch'].fillna('').str.lower().apply(
    lambda b: 'main'    if b in ('main', 'master') else
              'release' if b.startswith('release') else
              'feature' if b.startswith(('feat', 'feature', 'dev')) else
              'other'
)
means = df.groupby(branch_map)['duration_sec'].agg(['mean', 'median', 'count']).sort_values('count', ascending=False)
print(means)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(means.index, means['median'], color=[OK, INFO, ACCENT, WARN][:len(means)])
ax.set_ylabel('median duration (sec)')
ax.set_title('Median CI duration by branch class')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_3_3_branch_class.png', dpi=144)
plt.close(fig)

## 5. Hour-of-day patterns — runner contention proxy

In [ ]:
ts = pd.to_datetime(df['run_created_at'], utc=True, errors='coerce')
df_h = df.assign(hour=ts.dt.hour, dow=ts.dt.dayofweek).dropna(subset=['hour'])
hourly = df_h.groupby('hour')['duration_sec'].median()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(hourly.index, hourly.values, color=ACCENT, linewidth=2, marker='o')
ax.set_xlabel('hour of day (UTC)')
ax.set_ylabel('median duration (sec)')
ax.set_xticks(range(0, 24, 2))
ax.set_title('Median CI duration by hour of day')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_3_4_hour_of_day.png', dpi=144)
plt.close(fig)

## 6. Feature correlation matrix

Computed on the numeric features the model actually sees (post-transform).
Diagonal is always 1; off-diagonal magnitude indicates redundancy.

In [ ]:
from app.features.build import fit_schema, transform  # noqa: E402
schema = fit_schema(df)
X, names, _ = transform(df, schema)
# Take only numeric columns (last block) — categorical one-hots make
# the correlation matrix unreadable.
num_cols = schema.numeric_columns
num_idx = [names.index(c) for c in num_cols if c in names]
Xnum = X[:, num_idx]
corr = np.corrcoef(Xnum, rowvar=False)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha='right')
ax.set_yticks(range(len(num_cols)))
ax.set_yticklabels(num_cols)
fig.colorbar(im, ax=ax, label='Pearson r')
ax.set_title('Numeric feature correlation matrix')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_3_5_corr_matrix.png', dpi=144)
plt.close(fig)

## Summary

All figures land in `docs/thesis/figures/`. Suggested LaTeX inclusion:

```latex
\begin{figure}[h]
  \centering
  \includegraphics[width=0.85\textwidth]{figures/fig_3_1_duration_distribution.png}
  \caption{Распределение длительностей CI-задач (log-scale).}
  \label{fig:duration_dist}
\end{figure}
```